In [ ]:
# %%
# xyz2_filtered.ipynb - cell 1: setup, imports, configuration (noisy detector labels - with confidence filtering)

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import random
import joblib
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Optional, Any, Tuple, Dict
import json

# --- project root ---
# sets up the project's root path for local module imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# --- import from our csi_preprocessing module ---
# defines local helper functions for loading config and data paths
def load_app_config_local(project_root_path: Path, config_filename: str = "config.json") -> Dict:
    config_file = project_root_path / config_filename
    if not config_file.exists():
        raise FileNotFoundError(f"configuration file not found at {config_file}")
    with open(config_file, 'r') as f:
        return json.load(f)

def get_raw_data_path_local(project_root_path: Path, config_dict: dict, category_key: str, sub_category_key: str) -> Path:
    base_path = project_root_path / config_dict["data_dir"] / config_dict["raw_data_subdir"]
    try:
        category_path_part = config_dict["raw_data_categories"][category_key][sub_category_key]
    except KeyError:
        raise KeyError(f"category '{category_key}' or sub-category '{sub_category_key}' not found.")
    return base_path / category_path_part

def get_config_csi_metadata_cols_local(config_dict: dict) -> List[str]:
    return config_dict.get("csi_metadata_columns", ["people", "conf", "csi ts", "det ts"])

print("local helper functions for config/path loading defined.")

# --- load global configuration ---
print("\n--- loading global configuration ---")
try:
    config = load_app_config_local(PROJECT_ROOT)
    print("global configuration loaded successfully.")
    METADATA_COLS_FROM_CONFIG = get_config_csi_metadata_cols_local(config)
    OLD_PIPELINE_COLUMNS_TO_REMOVE = ["conf", "det ts", "csi ts"]
    TARGET_LABEL_COL = config.get("target_label_column", "people")
    DEAD_SUBSTITUTE_CARRIER_INDICES = config.get("dead_subcarrier_indices", [1,2,3,4,5,6,33,60,61,62,63,64])
    CONF_COL_NAME = config.get("csi_metadata_columns", ["people", "conf"])[1]
except Exception as e:
    print(f"error loading global configuration: {e}")
    raise

# --- this notebook uses noisy detector labels and applies confidence filtering ---
# USE_GROUND_TRUTH_LABELS_FLAG_FOR_CLEANING is false because we start with detector labels
# APPLY_CONFIDENCE_FILTER is true, which is the main purpose of this notebook
# CONFIDENCE_THRESHOLD is set to 0.6 for this experiment run
USE_GROUND_TRUTH_LABELS_FLAG_FOR_CLEANING = False
APPLY_CONFIDENCE_FILTER = True
CONFIDENCE_THRESHOLD = 0.7
# --- ----------------------------------------------------------------------- ---

EXPERIMENT_NAME = "Noisy_Detector_Labels_Conf_Filtered_Old_Pipeline_Rep"
EXPERIMENT_SUFFIX = "noisy_detector_conf_filtered" # unique suffix for this experiment

# --- define output paths for this experiment (xyz2_filtered) ---
# creates a unique set of directories for the filtered data experiment
BASE_OUTPUT_DIR_XYZ2_FILT = PROJECT_ROOT / f"experiment_outputs_{EXPERIMENT_SUFFIX}"
XYZ2_FILT_CLEAN_DIR = BASE_OUTPUT_DIR_XYZ2_FILT / "data_clean"
XYZ2_FILT_ARTIFACTS_DIR = BASE_OUTPUT_DIR_XYZ2_FILT / "artifacts"
XYZ2_FILT_WINDOWED_DATA_DIR = BASE_OUTPUT_DIR_XYZ2_FILT / "windowed_data"
XYZ2_FILT_TRAIN_WINDOW_DIR = XYZ2_FILT_WINDOWED_DATA_DIR / "train"
XYZ2_FILT_VAL_WINDOW_DIR = XYZ2_FILT_WINDOWED_DATA_DIR / "val"
XYZ2_FILT_TEST_WINDOW_DIR = XYZ2_FILT_WINDOWED_DATA_DIR / "test"
XYZ2_FILT_MODELS_DIR = PROJECT_ROOT / config["models_storage_base_dir"] / f"csi_detector_{EXPERIMENT_SUFFIX}"
XYZ2_FILT_PLOTS_DIR = PROJECT_ROOT / config["plots_base_dir"] / f"training_plots_{EXPERIMENT_SUFFIX}"

for dir_path in [XYZ2_FILT_CLEAN_DIR, XYZ2_FILT_ARTIFACTS_DIR, XYZ2_FILT_WINDOWED_DATA_DIR,
                 XYZ2_FILT_TRAIN_WINDOW_DIR, XYZ2_FILT_VAL_WINDOW_DIR, XYZ2_FILT_TEST_WINDOW_DIR,
                 XYZ2_FILT_MODELS_DIR, XYZ2_FILT_PLOTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\n--- paths for experiment: {EXPERIMENT_NAME} ---")
print(f"clean data (confidence filtered) dir: {XYZ2_FILT_CLEAN_DIR}")
print(f"artifacts dir: {XYZ2_FILT_ARTIFACTS_DIR}")
print(f"windowed data (final x,y) dir: {XYZ2_FILT_WINDOWED_DATA_DIR}")
print(f"trained models dir: {XYZ2_FILT_MODELS_DIR}")
print(f"training plots dir: {XYZ2_FILT_PLOTS_DIR}")

# sets pipeline parameters, identical to previous notebooks for consistency
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

DWT_BEFORE_PCA = True
DWT_WAVELET_OLD = 'db4'
DWT_LEVEL_OLD = 4
PCA_N_COMPONENTS_OLD = 0.99
PCA_SVD_SOLVER_OLD = "full"
PCA_RANDOM_STATE_OLD = SEED
WIN_SIZE_XYZ2_FILT = 16
STEP_SIZE_XYZ2_FILT = 8
MLP_EPOCHS_XYZ2_FILT = 60
MLP_BATCH_SIZE_XYZ2_FILT = 512
MLP_LEARNING_RATE_XYZ2_FILT = 3e-4
sns.set_theme(style="whitegrid", palette="pastel"); PLOT_DPI = 200

print("\n✔ cell 1: configuration for xyz2_filtered.ipynb done.")
print(f"apply_confidence_filter: {APPLY_CONFIDENCE_FILTER} with threshold {CONFIDENCE_THRESHOLD}")
print(f"dwt_before_pca set to: {DWT_BEFORE_PCA}")

In [ ]:
# %%
# xyz2_filtered.ipynb - cell 2: initial data cleaning (correcting low-confidence detections - robust conf normalization)

print(f"\n--- starting data cleaning for: {EXPERIMENT_NAME} (correcting low-confidence detections: {APPLY_CONFIDENCE_FILTER}) ---")
files_cleaned_count_xyz2_filt = 0
total_rows_processed_xyz2_filt = 0
low_confidence_detections_nulled_count = 0
rows_with_nan_conf_nulled_presence = 0

# helper to remove dead subcarriers, same as in the csi_preprocessing module
def remove_known_dead_csi_columns_local(df: pd.DataFrame, dead_indices: List[int]) -> pd.DataFrame:
    cols_to_remove = []
    for sc_idx in dead_indices:
        cols_to_remove.append(str(sc_idx))
        cols_to_remove.append(str(sc_idx) + "i")
    existing_to_remove = [col for col in cols_to_remove if col in df.columns]
    return df.drop(columns=existing_to_remove, errors='ignore')

if "raw_data_categories" not in config:
    raise KeyError("'raw_data_categories' not found in loaded config. ensure cell 1 ran successfully.")

# iterates through all raw csv files to apply filtering and cleaning
for category_key, sub_categories in config["raw_data_categories"].items():
    for sub_category_key, _ in sub_categories.items():
        current_raw_data_dir = get_raw_data_path_local(PROJECT_ROOT, config, category_key, sub_category_key)
        output_clean_subdir_xyz2_filt = XYZ2_FILT_CLEAN_DIR / category_key / sub_category_key
        output_clean_subdir_xyz2_filt.mkdir(parents=True, exist_ok=True)

        if not current_raw_data_dir.exists():
            print(f"warning: raw data directory not found: {current_raw_data_dir}")
            continue

        print(f"processing directory: {current_raw_data_dir}")

        for raw_csv_file_path in sorted(list(current_raw_data_dir.glob('*.csv'))):
            try:
                df_raw = pd.read_csv(raw_csv_file_path)
                if df_raw.empty:
                    print(f"  skipping {raw_csv_file_path.name}: raw file is empty.")
                    continue
                total_rows_processed_xyz2_filt += len(df_raw)
                df_processed = df_raw.copy()

                # this block applies the confidence filtering
                if APPLY_CONFIDENCE_FILTER:
                    if CONF_COL_NAME in df_processed.columns and TARGET_LABEL_COL in df_processed.columns:
                        # handles both float (0-1) and integer (0-1000) confidence scales by normalizing to 0-1
                        df_processed[CONF_COL_NAME] = pd.to_numeric(df_processed[CONF_COL_NAME], errors='coerce')
                        df_processed['conf_normalized'] = np.nan
                        valid_conf_mask = df_processed[CONF_COL_NAME].notna()
                        integer_scale_mask = valid_conf_mask & (df_processed[CONF_COL_NAME] > 1.0)
                        float_scale_mask = valid_conf_mask & (df_processed[CONF_COL_NAME] >= 0) & (df_processed[CONF_COL_NAME] <= 1.0)
                        df_processed.loc[integer_scale_mask, 'conf_normalized'] = df_processed.loc[integer_scale_mask, CONF_COL_NAME] / 1000.0
                        df_processed.loc[float_scale_mask, 'conf_normalized'] = df_processed.loc[float_scale_mask, CONF_COL_NAME]
                        df_processed['conf_normalized'] = np.clip(df_processed['conf_normalized'], 0.0, 1.0)
                        df_processed[TARGET_LABEL_COL] = pd.to_numeric(df_processed[TARGET_LABEL_COL], errors='coerce').fillna(0).astype(int)

                        # if a detection has a confidence below the threshold, its label is set to 0
                        condition_low_or_invalid_conf_presence = \
                            (df_processed[TARGET_LABEL_COL] > 0) & \
                            ( (df_processed['conf_normalized'] < CONFIDENCE_THRESHOLD) | df_processed['conf_normalized'].isna() )
                        num_to_null_this_file = condition_low_or_invalid_conf_presence.sum()
                        if num_to_null_this_file > 0:
                            low_confidence_detections_nulled_count += num_to_null_this_file
                        df_processed.loc[condition_low_or_invalid_conf_presence, TARGET_LABEL_COL] = 0
                        if 'conf_normalized' in df_processed.columns:
                             df_processed = df_processed.drop(columns=['conf_normalized'])
                    else:
                        print(f"warning: '{CONF_COL_NAME}' or '{TARGET_LABEL_COL}' not found in {raw_csv_file_path.name}. cannot apply confidence adjustment.")

                # drops other metadata columns after filtering
                df_cleaned_intermediate = df_processed.drop(columns=[c for c in OLD_PIPELINE_COLUMNS_TO_REMOVE if c in df_processed.columns], errors='ignore')

                if TARGET_LABEL_COL not in df_cleaned_intermediate.columns:
                    if TARGET_LABEL_COL in df_processed.columns:
                         df_cleaned_intermediate[TARGET_LABEL_COL] = df_processed[TARGET_LABEL_COL]
                    else:
                        print(f"error: target label column '{TARGET_LABEL_COL}' is missing for {raw_csv_file_path.name}. skipping.")
                        continue

                # removes the dead subcarriers
                df_final_to_save = remove_known_dead_csi_columns_local(df_cleaned_intermediate.copy(), DEAD_SUBSTITUTE_CARRIER_INDICES)

                output_file_path = output_clean_subdir_xyz2_filt / raw_csv_file_path.name
                df_final_to_save.to_csv(output_file_path, index=False)
                files_cleaned_count_xyz2_filt += 1
            except Exception as e:
                print(f"error cleaning file {raw_csv_file_path.name}: {e}")
                import traceback; traceback.print_exc()

print(f"\ntotal rows processed across all files: {total_rows_processed_xyz2_filt}")
if APPLY_CONFIDENCE_FILTER:
    print(f"total low-confidence/invalid-confidence 'presence' detections (people > 0 originally) nulled to '0': {low_confidence_detections_nulled_count}")

print(f"\n✔ cell 2: data cleaning for {EXPERIMENT_NAME} complete. {files_cleaned_count_xyz2_filt} files cleaned into '{XYZ2_FILT_CLEAN_DIR}'.")

In [ ]:
# %%
# xyz2_filtered.ipynb - cell 3: define "old pipeline" processing helper functions

# loads x and y from a csv that has already been confidence-filtered
def load_cleaned_xyz2_filt_csv(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    if TARGET_LABEL_COL not in df.columns:
        raise ValueError(f"target label column '{TARGET_LABEL_COL}' not found in {path}")
    X = df.drop(columns=[TARGET_LABEL_COL], errors='ignore').values.astype(np.float32)
    y = df[TARGET_LABEL_COL].values.astype(np.int64)
    return X, y

# applies dwt denoising, same as previous notebooks
def dwt_denoise_matrix_xyz2_filt(X: np.ndarray, wavelet: str = DWT_WAVELET_OLD, level: int = DWT_LEVEL_OLD) -> np.ndarray:
    n_rows, n_features = X.shape; Xd = np.zeros_like(X)
    if n_rows == 0: return Xd
    wavelet_obj = pywt.Wavelet(wavelet)
    for i in range(n_features):
        col = X[:, i]; current_max_level = pywt.dwt_max_level(len(col), wavelet_obj)
        actual_level = min(level, current_max_level)
        if actual_level < 1: Xd[:, i] = col; continue
        coeffs = pywt.wavedec(col, wavelet, level=actual_level)
        detail_coeffs_finest = coeffs[-1]; sigma = 0
        if len(detail_coeffs_finest) > 0: sigma = np.median(np.abs(detail_coeffs_finest - np.median(detail_coeffs_finest))) / 0.6745
        thresh = 0.0
        if sigma > 0 and n_rows > 1: thresh = sigma * np.sqrt(2 * np.log(n_rows))
        elif sigma > 0 and n_rows == 1: thresh = sigma
        coeffs_thresholded = [coeffs[0]] + [pywt.threshold(c, thresh, mode="soft") for c in coeffs[1:]]
        reconstructed_col = pywt.waverec(coeffs_thresholded, wavelet)
        if len(reconstructed_col) > n_rows: Xd[:, i] = reconstructed_col[:n_rows]
        elif len(reconstructed_col) < n_rows and len(reconstructed_col) > 0:
            Xd[:, i] = np.concatenate([reconstructed_col, np.full(n_rows - len(reconstructed_col), reconstructed_col[-1])])
        elif len(reconstructed_col) == 0 and n_rows > 0: Xd[:, i] = np.zeros(n_rows)
        else: Xd[:, i] = reconstructed_col
    return Xd

# applies a fitted pca model, same as previous notebooks
def apply_pca_xyz2_filt(X: np.ndarray, pca_model: PCA) -> np.ndarray:
    if not hasattr(pca_model, 'n_components_') or pca_model.n_components_ == 0: return X
    if X.shape[1] > 0 and X.shape[1] != pca_model.n_features_in_:
        print(f"pca warning: input features ({X.shape[1]}) != pca fit features ({pca_model.n_features_in_}). returning original.")
        return X
    elif X.shape[1] == 0 and pca_model.n_features_in_ > 0 :
        print(f"pca warning: input features are empty, but pca was fit on {pca_model.n_features_in_} features. returning empty array with {pca_model.n_components_} components.")
        return np.array([]).reshape(0, pca_model.n_components_)
    return pca_model.transform(X)

# creates 3d windows and maps the (filtered) detector labels to binary for each window
def window_data_xyz2_filt_3d(X: np.ndarray, y: np.ndarray, window_size: int, step_size: int) -> Tuple[np.ndarray, np.ndarray]:
    segments, labels = [], []
    N, num_features = X.shape
    if N < window_size or (num_features == 0 and N > 0):
        return np.array([]).reshape(0, window_size, num_features), np.array([])
    if N==0 : return np.array([]).reshape(0, window_size, num_features), np.array([])
    for start in range(0, N - window_size + 1, step_size):
        segments.append(X[start : start + window_size])
        label_segment = y[start : start + window_size]
        current_window_label = 0
        if len(label_segment) > 0:
            # if any non-zero label exists in the window, the window label is 1
            if any(val != 0 for val in label_segment):
                current_window_label = 1
            else:
                current_window_label = 0
        labels.append(current_window_label)
    if not segments:
        return np.array([]).reshape(0, window_size, num_features), np.array([])
    return np.stack(segments).astype(np.float32), np.array(labels).astype(np.int64)

print("✔ cell 3: helper functions for xyz2_filtered.ipynb defined.")

In [ ]:
# %%
# xyz2_filtered.ipynb - cell 4: stratified file splitting, scaler & pca fitting

print(f"\n--- starting stratified file splitting, scaler & pca fitting for: {EXPERIMENT_NAME} ---")

# 1. lists all cleaned (and filtered) csv files for the splitting process
files_by_sublocation_xyz2_filt = {}
print("\n--- debug: checking for cleaned files in sub-locations ---")
for category_key, sub_categories_map in config["raw_data_categories"].items():
    print(f"processing category_key from config: {category_key}")
    for sub_category_key, relative_path_part in sub_categories_map.items():
        sub_loc_key = f"{category_key}_{sub_category_key}"
        current_clean_sub_dir = XYZ2_FILT_CLEAN_DIR / category_key / sub_category_key
        print(f"  checking directory: {current_clean_sub_dir}")
        if current_clean_sub_dir.exists():
            files = sorted(list(current_clean_sub_dir.glob("*.csv")))
            if files:
                print(f"    found {len(files)} csv files. adding to files_by_sublocation_xyz2_filt for key '{sub_loc_key}'.")
                files_by_sublocation_xyz2_filt[sub_loc_key] = files
            else:
                print(f"    warning: no csv files found in {current_clean_sub_dir} although directory exists.")
        else:
            print(f"    warning: cleaned directory does not exist: {current_clean_sub_dir}")

if not files_by_sublocation_xyz2_filt:
    raise FileNotFoundError(f"no cleaned files found in any sub-location under {XYZ2_FILT_CLEAN_DIR}. ensure cell 2 (data cleaning) ran successfully and created output files.")

print("\nfiles successfully collected per sub-location for xyz2_filtered:")
for sub_loc, f_list in files_by_sublocation_xyz2_filt.items():
    print(f"  {sub_loc}: {len(f_list)} files")

# 2. splits the files into stratified train and test sets
train_files_xyz2_filt, test_files_xyz2_filt = [], []
np.random.seed(SEED)
for sub_loc_key, files_in_sub_loc in files_by_sublocation_xyz2_filt.items():
    num_files_in_sub_loc = len(files_in_sub_loc)
    if num_files_in_sub_loc == 0:
        print(f"warning: no files for sub_loc_key '{sub_loc_key}' during split. skipping.")
        continue
    n_test_for_sub_loc = 0
    if num_files_in_sub_loc > 1 :
        n_test_for_sub_loc = max(1, int(num_files_in_sub_loc * 0.20))
        if (num_files_in_sub_loc - n_test_for_sub_loc) < 1:
             n_test_for_sub_loc = num_files_in_sub_loc - 1
    elif num_files_in_sub_loc == 1:
        print(f"warning: only 1 file in {sub_loc_key}. assigning to training set to ensure data for fitting scaler/pca.")
        n_test_for_sub_loc = 0
    if n_test_for_sub_loc < 0 : n_test_for_sub_loc = 0
    chosen_test_files = list(np.random.choice(files_in_sub_loc, n_test_for_sub_loc, replace=False)) if n_test_for_sub_loc > 0 else []
    chosen_train_files = [f for f in files_in_sub_loc if f not in chosen_test_files]
    if not chosen_train_files and chosen_test_files:
        print(f"warning: all files from {sub_loc_key} went to test. moving one to train if possible.")
        if chosen_test_files:
             chosen_train_files.append(chosen_test_files.pop())
    test_files_xyz2_filt.extend(chosen_test_files)
    train_files_xyz2_filt.extend(chosen_train_files)

total_files_after_split = sum(len(v) for v in files_by_sublocation_xyz2_filt.values())
print(f"\ntotal cleaned files considered for splitting: {total_files_after_split}")
print(f"{EXPERIMENT_NAME} train files count: {len(train_files_xyz2_filt)}")
print(f"{EXPERIMENT_NAME} test files count: {len(test_files_xyz2_filt)}")
print("\ntest set breakdown by sub-location:")
test_counts_xyz2_filt = {}
for f_path in test_files_xyz2_filt:
    sub_loc_key = f"{f_path.parent.parent.name}_{f_path.parent.name}"
    test_counts_xyz2_filt[sub_loc_key] = test_counts_xyz2_filt.get(sub_loc_key, 0) + 1
if test_counts_xyz2_filt:
    for sub_loc, count in sorted(test_counts_xyz2_filt.items()): print(f"  {sub_loc}: {count}")
else:
    print("  test set is empty.")

# 3. stacks the training data to fit the scaler and pca model
X_stack_xyz2_filt_list, y_stack_xyz2_filt_detector_labels = [], []
if not train_files_xyz2_filt:
    print("warning: no files in the training set after splitting. cannot fit scaler/pca.")
    X_stack_xyz2_filt = np.array([]).reshape(0,1)
    y_stack_xyz2_filt_orig_labels = np.array([])
    scaler_xyz2_filt = StandardScaler()
    pca_xyz2_filt = PCA(n_components=0)
    joblib.dump(scaler_xyz2_filt, XYZ2_FILT_ARTIFACTS_DIR / f"scaler_{EXPERIMENT_SUFFIX}.joblib")
    joblib.dump(pca_xyz2_filt, XYZ2_FILT_ARTIFACTS_DIR / f"pca_{EXPERIMENT_SUFFIX}.joblib")
    print(f"dummy scaler and pca model saved for {EXPERIMENT_SUFFIX} as training set was empty.")
else:
    for p_path in train_files_xyz2_filt:
        X_f, y_f_detector = load_cleaned_xyz2_filt_csv(p_path)
        if X_f.shape[0] > 0:
            X_stack_xyz2_filt_list.append(X_f)
            y_stack_xyz2_filt_detector_labels.append(y_f_detector)
    if not X_stack_xyz2_filt_list:
        raise ValueError(f"no data loaded from {EXPERIMENT_NAME} training files after filtering empty files. cannot fit scaler/pca.")
    X_stack_xyz2_filt = np.vstack(X_stack_xyz2_filt_list)
    y_stack_xyz2_filt_orig_labels = np.concatenate(y_stack_xyz2_filt_detector_labels)
    print(f"\n{EXPERIMENT_NAME} x_stack shape (for scaler/pca): {X_stack_xyz2_filt.shape}")
    unique_orig_lbl_tr, counts_orig_lbl_tr = np.unique(y_stack_xyz2_filt_orig_labels, return_counts=True)
    print(f"original detector label distribution in {EXPERIMENT_NAME} training stack: {dict(zip(unique_orig_lbl_tr, counts_orig_lbl_tr))}")

    # 4. fits and saves the standardscaler
    scaler_xyz2_filt = StandardScaler().fit(X_stack_xyz2_filt)
    scaler_save_path_xyz2_filt = XYZ2_FILT_ARTIFACTS_DIR / f"scaler_{EXPERIMENT_SUFFIX}.joblib"
    joblib.dump(scaler_xyz2_filt, scaler_save_path_xyz2_filt)
    print(f"{EXPERIMENT_NAME} scaler saved: {scaler_save_path_xyz2_filt}")

    # 5. scales, applies dwt, and then fits and saves the pca model
    Xn_xyz2_filt = scaler_xyz2_filt.transform(X_stack_xyz2_filt)
    X_for_pca_fit_xyz2_filt = Xn_xyz2_filt
    if DWT_BEFORE_PCA:
        print(f"applying dwt before pca to {EXPERIMENT_NAME} training stack...")
        X_for_pca_fit_xyz2_filt = dwt_denoise_matrix_xyz2_filt(Xn_xyz2_filt.copy())
    else:
        print(f"skipping dwt before pca for {EXPERIMENT_NAME} (dwt applied later to pcs per file).")
    if X_for_pca_fit_xyz2_filt.shape[1] == 0:
        print(f"warning: no features available to fit pca for {EXPERIMENT_NAME}. pca will be null.")
        pca_xyz2_filt = PCA(n_components=0)
    else:
        pca_xyz2_filt = PCA(n_components=PCA_N_COMPONENTS_OLD, svd_solver=PCA_SVD_SOLVER_OLD, random_state=PCA_RANDOM_STATE_OLD)
        pca_xyz2_filt.fit(X_for_pca_fit_xyz2_filt)
    pca_save_path_xyz2_filt = XYZ2_FILT_ARTIFACTS_DIR / f"pca_{EXPERIMENT_SUFFIX}.joblib"
    joblib.dump(pca_xyz2_filt, pca_save_path_xyz2_filt)
    print(f"{EXPERIMENT_NAME} pca model saved: {pca_save_path_xyz2_filt}")
    n_pca_comp_xyz2_filt = getattr(pca_xyz2_filt, 'n_components_', 0)
    print(f"  pca components fitted: {n_pca_comp_xyz2_filt}")
    if n_pca_comp_xyz2_filt > 0 and hasattr(pca_xyz2_filt, 'explained_variance_ratio_') and pca_xyz2_filt.explained_variance_ratio_ is not None:
        print(f"  explained variance: {np.sum(pca_xyz2_filt.explained_variance_ratio_):.4f}")

print(f"\n✔ cell 4: scaler and pca for {EXPERIMENT_NAME} fitted and saved.")

In [ ]:
# %%
# xyz2_filtered.ipynb - cell 5: windowing and data processing for each split

print(f"\n--- starting windowing for: {EXPERIMENT_NAME} ---")

scaler_loaded_xyz2_filt = joblib.load(XYZ2_FILT_ARTIFACTS_DIR / f"scaler_{EXPERIMENT_SUFFIX}.joblib")
pca_loaded_xyz2_filt = joblib.load(XYZ2_FILT_ARTIFACTS_DIR / f"pca_{EXPERIMENT_SUFFIX}.joblib")
print(f"reloaded scaler and pca model for {EXPERIMENT_SUFFIX}.")
print(f"using fixed win_size_xyz2_filt={WIN_SIZE_XYZ2_FILT}, step_size_xyz2_filt={STEP_SIZE_XYZ2_FILT}")

apply_dwt_after_pca_flag_xyz2_filt = not DWT_BEFORE_PCA

# applies the full processing pipeline to a list of files
def process_data_split_xyz2_filt(files_list: List[Path],
                                scaler: StandardScaler, pca_model: PCA,
                                win_sz: int, step_sz: int,
                                dwt_after_pca: bool, dwt_before_pca: bool):
    print(f"processing {len(files_list)} files for windowing...")
    X_all_win, y_all_win = [], []
    for p_path in files_list:
        X_f, y_f_orig_detector = load_cleaned_xyz2_filt_csv(p_path)
        if X_f.shape[0] < win_sz: continue
        # applies the fitted transformations: scale -> dwt -> pca
        Xn = scaler.transform(X_f)
        X_pre_pca = Xn
        if dwt_before_pca: X_pre_pca = dwt_denoise_matrix_xyz2_filt(Xn.copy())
        Xp = apply_pca_xyz2_filt(X_pre_pca, pca_model)
        X_final = Xp
        if dwt_after_pca: X_final = dwt_denoise_matrix_xyz2_filt(Xp.copy())
        # windows the data and creates binary labels
        Xw, yw = window_data_xyz2_filt_3d(X_final, y_f_orig_detector,
                                          win_sz, step_sz
        )
        if Xw.size > 0: X_all_win.append(Xw); y_all_win.append(yw)
    if not X_all_win:
        n_feat = getattr(pca_model, 'n_components_', X_f.shape[1] if 'X_f' in locals() and X_f is not None else 1)
        return np.array([]).reshape(0, win_sz, n_feat if n_feat > 0 else 1), np.array([])
    return np.concatenate(X_all_win, axis=0), np.concatenate(y_all_win, axis=0)

# creates a validation set from the training files
stratify_labels_for_val_split_xyz2_filt = [1 if "1_pers" in str(p).lower() else 0 for p in train_files_xyz2_filt]
can_stratify_val_xyz2_filt = len(np.unique(stratify_labels_for_val_split_xyz2_filt)) > 1 and \
                   all(np.sum(np.array(stratify_labels_for_val_split_xyz2_filt) == lbl) >= 2
                       for lbl in np.unique(stratify_labels_for_val_split_xyz2_filt))
val_split_size_xyz2_filt = 0.15

if len(train_files_xyz2_filt) * val_split_size_xyz2_filt < 1 or not can_stratify_val_xyz2_filt :
    train_sub_files_xyz2_filt = train_files_xyz2_filt; val_files_xyz2_filt = []
else:
    try:
        train_sub_files_xyz2_filt, val_files_xyz2_filt = train_test_split(
            train_files_xyz2_filt, test_size=val_split_size_xyz2_filt,
            stratify=stratify_labels_for_val_split_xyz2_filt, random_state=SEED, shuffle=True
        )
    except ValueError:
        train_sub_files_xyz2_filt, val_files_xyz2_filt = train_test_split(train_files_xyz2_filt, test_size=val_split_size_xyz2_filt, random_state=SEED, shuffle=True)

print(f"{EXPERIMENT_NAME} final training files: {len(train_sub_files_xyz2_filt)}")
print(f"{EXPERIMENT_NAME} final validation files: {len(val_files_xyz2_filt)}")

# processes the train, validation, and test sets
X_train_xyz2_filt, y_train_xyz2_filt = process_data_split_xyz2_filt(
    train_sub_files_xyz2_filt, scaler_loaded_xyz2_filt, pca_loaded_xyz2_filt,
    WIN_SIZE_XYZ2_FILT, STEP_SIZE_XYZ2_FILT, apply_dwt_after_pca_flag_xyz2_filt, DWT_BEFORE_PCA
)
if val_files_xyz2_filt:
    X_val_xyz2_filt, y_val_xyz2_filt = process_data_split_xyz2_filt(
        val_files_xyz2_filt, scaler_loaded_xyz2_filt, pca_loaded_xyz2_filt,
        WIN_SIZE_XYZ2_FILT, STEP_SIZE_XYZ2_FILT, apply_dwt_after_pca_flag_xyz2_filt, DWT_BEFORE_PCA
    )
else:
    n_feat_val_xyz2_filt = getattr(pca_loaded_xyz2_filt, 'n_components_', X_stack_xyz2_filt.shape[1] if 'X_stack_xyz2_filt' in locals() else 1)
    X_val_xyz2_filt, y_val_xyz2_filt = np.array([]).reshape(0, WIN_SIZE_XYZ2_FILT, n_feat_val_xyz2_filt if n_feat_val_xyz2_filt > 0 else 1), np.array([])
X_test_xyz2_filt, y_test_xyz2_filt = process_data_split_xyz2_filt(
    test_files_xyz2_filt, scaler_loaded_xyz2_filt, pca_loaded_xyz2_filt,
    WIN_SIZE_XYZ2_FILT, STEP_SIZE_XYZ2_FILT, apply_dwt_after_pca_flag_xyz2_filt, DWT_BEFORE_PCA
)

# saves the final windowed numpy arrays
np.save(XYZ2_FILT_TRAIN_WINDOW_DIR / "X_train_windows.npy", X_train_xyz2_filt)
np.save(XYZ2_FILT_TRAIN_WINDOW_DIR / "y_train_labels.npy", y_train_xyz2_filt)
np.save(XYZ2_FILT_VAL_WINDOW_DIR / "X_val_windows.npy", X_val_xyz2_filt)
np.save(XYZ2_FILT_VAL_WINDOW_DIR / "y_val_labels.npy", y_val_xyz2_filt)
np.save(XYZ2_FILT_TEST_WINDOW_DIR / "X_test_windows.npy", X_test_xyz2_filt)
np.save(XYZ2_FILT_TEST_WINDOW_DIR / "y_test_labels.npy", y_test_xyz2_filt)
print(f"final windowed data for {EXPERIMENT_NAME} saved to {XYZ2_FILT_WINDOWED_DATA_DIR}")

print(f"\n{EXPERIMENT_NAME} processed train data -> x:{X_train_xyz2_filt.shape}, y:{y_train_xyz2_filt.shape}")
if y_train_xyz2_filt.size > 0: print(f"  unique y_train_xyz2_filt labels (mapped to binary): {np.unique(y_train_xyz2_filt, return_counts=True)}")
print(f"dwt was applied {'before' if DWT_BEFORE_PCA else 'after'} pca for this {EXPERIMENT_NAME} run.")
print(f"labels used for windowing: detector mapped to binary (after confidence filter)")
print(f"\n✔ cell 5: windowing complete for {EXPERIMENT_NAME}.")

In [ ]:
# %%
# xyz2_filtered.ipynb - cell 6: mlp training and evaluation

print(f"\n--- starting mlp training & evaluation for: {EXPERIMENT_NAME} ---")

if 'X_train_xyz2_filt' not in locals() or X_train_xyz2_filt.size == 0 or \
   'X_val_xyz2_filt' not in locals() or \
   'X_test_xyz2_filt' not in locals() or X_test_xyz2_filt.size == 0:
    print(f"error: training, validation, or test data for {EXPERIMENT_NAME} is empty or not defined. skipping mlp training.")
else:
    # flattens the 3d windowed data into 2d for the mlp
    def flatten_xyz2_filt(X_3d: np.ndarray) -> np.ndarray:
        if X_3d.ndim == 3:
            n_s, n_t, n_f = X_3d.shape
            if n_f == 0 and n_s > 0: return np.array([]).reshape(n_s, 0)
            return X_3d.reshape(n_s, n_t * n_f).astype(np.float32)
        return X_3d.astype(np.float32)

    X_tr_flat_xyz2_filt = flatten_xyz2_filt(X_train_xyz2_filt)
    X_va_flat_xyz2_filt = flatten_xyz2_filt(X_val_xyz2_filt) if X_val_xyz2_filt.size > 0 else np.array([]).reshape(0, X_tr_flat_xyz2_filt.shape[1] if X_tr_flat_xyz2_filt.size > 0 else 0)
    X_test_flat_xyz2_filt = flatten_xyz2_filt(X_test_xyz2_filt)

    if X_tr_flat_xyz2_filt.shape[0] > 0 and X_tr_flat_xyz2_filt.shape[1] == 0 and X_train_xyz2_filt.shape[0] > 0 and X_train_xyz2_filt.shape[2] > 0 :
        print("error: input dimension for mlp is 0. cannot proceed.")
    elif X_tr_flat_xyz2_filt.shape[0] == 0 :
        print(f"error: training data for {EXPERIMENT_NAME} is empty. cannot proceed.")
    else:
        input_dim_xyz2_filt = X_tr_flat_xyz2_filt.shape[1]
        print(f"{EXPERIMENT_NAME} mlp input dimension: {input_dim_xyz2_filt}")

        # calculates class weights for the training data
        cw_dict_xyz2_filt = None
        if len(y_train_xyz2_filt) > 0 and len(np.unique(y_train_xyz2_filt)) > 1:
            try:
                cw_vals_xyz2_filt = class_weight.compute_class_weight(class_weight="balanced", classes=np.unique(y_train_xyz2_filt), y=y_train_xyz2_filt)
                cw_dict_xyz2_filt = {int(c): float(w) for c, w in zip(np.unique(y_train_xyz2_filt), cw_vals_xyz2_filt)}
            except ValueError: cw_dict_xyz2_filt = None
        print(f"{EXPERIMENT_NAME} class weights: {cw_dict_xyz2_filt}")

        # defines the mlp architecture, same as the one in the paper
        def build_mlp_xyz2_filt(dim, name_suffix=""):
            inp = layers.Input(shape=(dim,))
            x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(inp)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.3)(x)
            x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.3)(x)
            x = layers.Dense(64,  kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(0.2)(x)
            x = layers.Dense(32,  kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
            out = layers.Dense(1, activation="sigmoid")(x)
            model = tf.keras.Model(inp, out, name=f"MLP_{EXPERIMENT_NAME}{name_suffix}")
            model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LEARNING_RATE_XYZ2_FILT), loss="binary_crossentropy", metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
            return model

        model_suffix_detail_xyz2_filt = f"_W{WIN_SIZE_XYZ2_FILT}_S{STEP_SIZE_XYZ2_FILT}_DWT_{'BeforePCA' if DWT_BEFORE_PCA else 'AfterPCA'}"
        mlp_xyz2_filt_model_name = f"MLP_{EXPERIMENT_NAME}{model_suffix_detail_xyz2_filt}"

        mlp_xyz2_filt = build_mlp_xyz2_filt(input_dim_xyz2_filt, name_suffix=model_suffix_detail_xyz2_filt)
        print(f"\n--- training {mlp_xyz2_filt_model_name} ---")
        mlp_xyz2_filt.summary()

        model_checkpoint_path_xyz2_filt = str(XYZ2_FILT_MODELS_DIR / mlp_xyz2_filt_model_name / "best_model.keras")
        (XYZ2_FILT_MODELS_DIR / mlp_xyz2_filt_model_name).mkdir(parents=True, exist_ok=True)

        validation_data_fit_xyz2_filt = None
        if X_va_flat_xyz2_filt.size > 0 and y_val_xyz2_filt.size > 0:
            validation_data_fit_xyz2_filt = (X_va_flat_xyz2_filt, y_val_xyz2_filt.astype(np.int32))

        # trains the model on the confidence-filtered data
        history_xyz2_filt = mlp_xyz2_filt.fit(
            X_tr_flat_xyz2_filt, y_train_xyz2_filt.astype(np.int32),
            epochs=MLP_EPOCHS_XYZ2_FILT, batch_size=MLP_BATCH_SIZE_XYZ2_FILT,
            validation_data=validation_data_fit_xyz2_filt,
            class_weight=cw_dict_xyz2_filt,
            callbacks=[
                callbacks.ReduceLROnPlateau(monitor='val_loss' if validation_data_fit_xyz2_filt else 'loss', patience=10, factor=0.5, min_lr=1e-6, verbose=1),
                callbacks.EarlyStopping(monitor='val_loss' if validation_data_fit_xyz2_filt else 'loss', patience=15, restore_best_weights=True, verbose=1),
                callbacks.ModelCheckpoint(filepath=model_checkpoint_path_xyz2_filt, save_best_only=True,
                                          monitor='val_accuracy' if validation_data_fit_xyz2_filt else 'accuracy',
                                          mode='max', verbose=1)
            ], verbose=2
        )

        if Path(model_checkpoint_path_xyz2_filt).exists(): mlp_xyz2_filt.load_weights(model_checkpoint_path_xyz2_filt)

        # evaluates the final model on the unseen test set
        print(f"\n--- evaluating {mlp_xyz2_filt_model_name} on test set ---")
        eval_results_xyz2_filt = mlp_xyz2_filt.evaluate(X_test_flat_xyz2_filt, y_test_xyz2_filt.astype(np.int32), verbose=0)
        y_prob_xyz2_filt = mlp_xyz2_filt.predict(X_test_flat_xyz2_filt).ravel()
        y_pred_xyz2_filt = (y_prob_xyz2_filt > 0.5).astype(int)

        print(f"\n{mlp_xyz2_filt_model_name} - test results:")
        print(f"  loss: {eval_results_xyz2_filt[0]:.4f}"); print(f"  accuracy: {eval_results_xyz2_filt[1]:.4f}")
        print(f"  precision (keras): {eval_results_xyz2_filt[2]:.4f}"); print(f"  recall (keras): {eval_results_xyz2_filt[3]:.4f}")
        f1_xyz2_filt = f1_score(y_test_xyz2_filt,y_pred_xyz2_filt,pos_label=1,average='binary',zero_division=0)
        prec_sk_xyz2_filt = precision_score(y_test_xyz2_filt,y_pred_xyz2_filt,pos_label=1,average='binary',zero_division=0)
        rec_sk_xyz2_filt = recall_score(y_test_xyz2_filt,y_pred_xyz2_filt,pos_label=1,average='binary',zero_division=0)
        print(f"  f1-score (sklearn class 1): {f1_xyz2_filt:.4f}")
        print(f"  precision (sklearn class 1): {prec_sk_xyz2_filt:.4f}")
        print(f"  recall (sklearn class 1): {rec_sk_xyz2_filt:.4f}")
        roc_auc_xyz2_filt = float('nan')
        if len(np.unique(y_test_xyz2_filt)) > 1:
            try: roc_auc_xyz2_filt = roc_auc_score(y_test_xyz2_filt, y_prob_xyz2_filt)
            except ValueError: pass
        if np.isnan(roc_auc_xyz2_filt): print("  roc auc: could not be calculated")
        else: print(f"  roc auc: {roc_auc_xyz2_filt:.4f}")

        mlp_xyz2_filt.save(XYZ2_FILT_MODELS_DIR / f"{mlp_xyz2_filt_model_name}_final.keras")
        print(f"✔ {mlp_xyz2_filt_model_name} (final) saved.")

        # generates and saves evaluation plots
        xyz2_filt_plot_save_dir = XYZ2_FILT_PLOTS_DIR / mlp_xyz2_filt_model_name
        xyz2_filt_plot_save_dir.mkdir(parents=True, exist_ok=True)

        def plot_training_history_abc1(history, model_name_str, save_dir):
            plt.figure(figsize=(14, 6)); plt.subplot(1, 2, 1)
            if 'accuracy' in history.history: plt.plot(history.history['accuracy'], label='train acc', marker='.')
            if 'val_accuracy' in history.history: plt.plot(history.history['val_accuracy'], label='val acc', marker='.')
            plt.title('accuracy', fontsize=14); plt.xlabel('epoch'); plt.ylabel('accuracy')
            if 'accuracy' in history.history or 'val_accuracy' in history.history: plt.legend()
            plt.grid(True, alpha=0.7); plt.subplot(1, 2, 2)
            if 'loss' in history.history: plt.plot(history.history['loss'], label='train loss', marker='.')
            if 'val_loss' in history.history: plt.plot(history.history['val_loss'], label='val loss', marker='.')
            plt.title('loss', fontsize=14); plt.xlabel('epoch'); plt.ylabel('loss')
            if 'loss' in history.history or 'val_loss' in history.history: plt.legend()
            plt.grid(True, alpha=0.7); plt.suptitle(f"training: {model_name_str}", fontsize=16, fontweight='bold')
            plt.tight_layout(rect=[0,0.03,1,0.95]); plt.savefig(save_dir/"th.png",dpi=PLOT_DPI); plt.close(); print(f"plot: {save_dir/'th.png'}")
        def plot_confusion_matrix_abc1(y_true, y_pred, model_name_str, class_names, save_dir):
            cm=confusion_matrix(y_true,y_pred); plt.figure(figsize=(7,6))
            sns.heatmap(cm,annot=True,fmt='d',cmap='blues',xticklabels=class_names,yticklabels=class_names,annot_kws={"size":16},cbar=False)
            plt.title(f'{model_name_str} - cm',fontsize=16,fontweight='bold');plt.xlabel('predicted');plt.ylabel('actual');plt.tight_layout()
            plt.savefig(save_dir/"cm.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'cm.png'}")
        def plot_roc_curve_abc1(y_true, y_prob, model_name_str, auc, save_dir):
            fpr,tpr,_=roc_curve(y_true,y_prob);plt.figure(figsize=(8,7))
            plt.plot(fpr,tpr,label=f'auc = {auc:.4f}',color='darkorange',lw=2.5);plt.plot([0,1],[0,1],'k--',lw=2)
            plt.xlim([-0.01,1.0]);plt.ylim([0.0,1.01]);plt.xlabel('fpr');plt.ylabel('tpr')
            plt.title(f'{model_name_str} - roc',fontsize=16,fontweight='bold');plt.legend(loc="lower right");plt.grid(True,alpha=0.7);plt.tight_layout()
            plt.savefig(save_dir/"roc.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'roc.png'}")
        def plot_pr_curve_abc1(y_true, y_prob, model_name_str, save_dir):
            prec,rec,_=precision_recall_curve(y_true,y_prob);plt.figure(figsize=(8,7))
            plt.plot(rec,prec,label='pr curve',color='mediumblue',lw=2.5)
            plt.xlabel('recall');plt.ylabel('precision');plt.ylim([0.0,1.01]);plt.xlim([0.0,1.0])
            plt.title(f'{model_name_str} - pr curve',fontsize=16,fontweight='bold');plt.grid(True,alpha=0.7);plt.tight_layout()
            plt.savefig(save_dir/"pr_curve.png",dpi=PLOT_DPI);plt.close(); print(f"plot: {save_dir/'pr_curve.png'}")

        if 'plot_training_history_abc1' in globals():
            plot_training_history_abc1(history_xyz2_filt, mlp_xyz2_filt_model_name, xyz2_filt_plot_save_dir)
            plot_confusion_matrix_abc1(y_test_xyz2_filt, y_pred_xyz2_filt, mlp_xyz2_filt_model_name, ["no person (0)", "person (1)"], xyz2_filt_plot_save_dir)
            if not np.isnan(roc_auc_xyz2_filt):
                plot_roc_curve_abc1(y_test_xyz2_filt, y_prob_xyz2_filt, mlp_xyz2_filt_model_name, roc_auc_xyz2_filt, xyz2_filt_plot_save_dir)
            plot_pr_curve_abc1(y_test_xyz2_filt, y_prob_xyz2_filt, mlp_xyz2_filt_model_name, xyz2_filt_plot_save_dir)
        else:
            print("local plotting functions not found. skipping result plots.")

        print(f"\n✔ cell 6 for {EXPERIMENT_NAME}: {mlp_xyz2_filt_model_name} training and evaluation complete.")